# 귀인요인 설문 전체 통계분석

**분석순서:** 데이터로드 → 요인분석(SPSS비교) → 변수계산 → 신뢰도 → 기술통계+정규분포 → 상관관계 → 회귀분석 → 기울기그래프 → 해석

In [ ]:
%config InlineBackend.figure_format = "retina"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import pingouin as pg
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ── 데이터 로드 ──────────────────────────────────────────
# CSV 파일을 안티그래비티 폴더에 넣고 파일명 입력
FILE = 'csv_1분석전용_귀인요인.csv'
try:
    df = pd.read_csv(FILE, encoding='cp949')
except:
    df = pd.read_csv(FILE, encoding='utf-8')

# 컬럼명 공백·줄바꿈 제거
df.columns = df.columns.str.strip().str.replace('\n','')

print(f'✅ 데이터 로드 완료: {df.shape[0]}명, {df.shape[1]}개 변수')
print('컬럼목록:', list(df.columns))
df.head(3)

---
## STEP 1. 요인분석 (SPSS 결과와 비교)

### 1-1. A그룹 + FCYN 요인분석

In [ ]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

def run_fa(data, cols, n_factors, title):
    d = data[cols].dropna()
    kmo_all, kmo_m = calculate_kmo(d)
    chi2, p = calculate_bartlett_sphericity(d)
    print(f'\n{'='*60}')
    print(f'  {title}')
    print(f'{'='*60}')
    print(f'  KMO = {kmo_m:.3f}  (≥0.6 적합)')
    print(f'  Bartlett χ²={chi2:.1f}, p={p:.4f}  (p<.05 유의)')

    fa = FactorAnalyzer(n_factors=n_factors, rotation='varimax')
    fa.fit(d)

    fnames = [f'요인{i+1}' for i in range(n_factors)]
    loadings = pd.DataFrame(fa.loadings_, index=cols, columns=fnames).round(3)

    # 0.4 이상 굵게 표시
    print('\n  ▶ 요인부하량 (|값|≥0.4 → 유의)')
    print(f'  {"-"*50}')
    print(f'  {"변수":<10}', end='')
    for f in fnames: print(f'{f:>10}', end='')
    print()
    for idx, row in loadings.iterrows():
        print(f'  {idx:<10}', end='')
        for v in row:
            mark = '***' if abs(v)>=0.4 else '   '
            print(f'{v:>7.3f}{mark}', end='')
        print()

    ev, _ = fa.get_eigenvalues()
    var = fa.get_factor_variance()
    print(f'\n  ▶ 분산 설명량')
    for i, f in enumerate(fnames):
        print(f'  {f}: 고유값={ev[i]:.3f}, 분산={var[1][i]*100:.1f}%, 누적={var[2][i]*100:.1f}%')
    return loadings

# ── A그룹 + FCYN (3요인) ──────────────────────────
A_FCYN_COLS = ['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10','A11','A12',
               'FCYN1','FCYN2','FCYN3','FCYN4']
existing_A = [c for c in A_FCYN_COLS if c in df.columns]
fa1 = run_fa(df, existing_A, 3, 'A그룹 + FCYN 요인분석 (3요인, Varimax)')

In [ ]:
# ── B, DT, EN 그룹 요인분석 ──────────────────────
B_DT_EN_COLS = ['B13','B14','B15','B16','B17','B18',
                'DT1','DT2','DT3','DT4','DT5','DT6',
                'EN1','EN2','EN3','EN4','EN5','EN6']
existing_B = [c for c in B_DT_EN_COLS if c in df.columns]
fa2 = run_fa(df, existing_B, 3, 'B + DT + EN 요인분석 (3요인, Varimax)')

In [ ]:
# ── C그룹 요인분석 (참고용 - SPSS에서도 안됨) ────
C_COLS = ['c19','c20','c21','c22','c23','c24']
existing_C = [c for c in C_COLS if c in df.columns]
if existing_C:
    fa3 = run_fa(df, existing_C, 2, 'C그룹 요인분석 (참고용)')
    print('\n  → SPSS 결과와 동일하게 C그룹은 단일요인으로 묶이지 않음 → 변수계산 제외')

---
## STEP 2. 변수계산 (잠재변수 생성)

SPSS 결과 기반으로 확정된 문항 묶음

In [ ]:
# ── SPSS 결과 기반 잠재변수 정의 ─────────────────────────
LATENT = {
    # A그룹
    '내적귀인': ['A1','A2','A3','A5','A6','A7','A10'],      # 요인1
    '통제위인': ['FCYN1','FCYN2','FCYN3','FCYN4'],            # 요인2
    '외적귀인': ['A9','A11','A12'],                           # 요인3
    # B, DT, EN 그룹
    '행동요인_B':  ['B13','B14','B15','B16','B17'],           # 요인1
    '동기요인_DT': ['DT1','DT2','DT3','DT5','DT6'],           # 요인1
    '환경요인_EN1':['EN1','EN2','EN3','EN4'],                  # 요인2
    '환경요인_EN2':['EN5','EN6'],                              # 요인3
    # C 제외
}

print('▶ 잠재변수 생성 결과')
print('='*50)
for name, cols in LATENT.items():
    exist = [c for c in cols if c in df.columns]
    if exist:
        df[name] = df[exist].mean(axis=1)
        m = df[name].mean()
        s = df[name].std()
        print(f'  {name:<15}: 문항수={len(exist)}, 평균={m:.3f}, SD={s:.3f}')
    else:
        print(f'  {name}: 컬럼 없음')

LV = list(LATENT.keys())
print(f'\n✅ 잠재변수 {len(LV)}개 생성 완료')

---
## STEP 3. 신뢰도 분석 (Cronbach α)

In [ ]:
print('▶ 신뢰도 분석 (Cronbach α)')
print('='*55)
print(f'  기준: α≥.90 우수 | α≥.80 양호 | α≥.70 수용 | α<.60 부적합')
print('='*55)

alpha_results = {}
for name, cols in LATENT.items():
    exist = [c for c in cols if c in df.columns]
    if len(exist) >= 2:
        a, ci = pg.cronbach_alpha(df[exist].dropna())
        grade = '우수' if a>=.9 else '양호' if a>=.8 else '수용' if a>=.7 else '부적합'
        alpha_results[name] = a
        print(f'  {name:<15}: α = {a:.3f}  95%CI [{ci[0]:.3f},{ci[1]:.3f}]  → {grade}')

print('\n  ✅ α≥.70 기준 모두 통과한 변수만 분석에 사용')

---
## STEP 4. 기술통계 + 정규분포 검정

In [ ]:
valid_LV = [v for v in LV if v in df.columns]
desc = df[valid_LV].describe().T
desc['왜도'] = df[valid_LV].skew()
desc['첨도'] = df[valid_LV].kurtosis()

# Shapiro-Wilk 정규성 검정
sw_p = []
for v in valid_LV:
    _, p = stats.shapiro(df[v].dropna())
    sw_p.append(p)
desc['Shapiro p'] = sw_p
desc['정규성'] = ['정규분포' if p>.05 else '비정규' for p in sw_p]

desc.columns = ['n','평균','표준편차','최솟값','Q1','중앙값','Q3','최댓값','왜도','첨도','SW_p','정규성']
print('▶ 기술통계 + 정규성 검정 결과')
print(desc[['n','평균','표준편차','왜도','첨도','SW_p','정규성']].round(3).to_string())
print('\n  * 왜도|<2, |첨도|<7 이면 정규분포 가정 가능 (Hair et al., 2010)')

In [ ]:
# ── 정규분포 그래프 ──────────────────────────────
n = len(valid_LV)
cols_n = 3
rows_n = (n + cols_n - 1) // cols_n

fig, axes = plt.subplots(rows_n, cols_n, figsize=(5*cols_n, 4*rows_n))
axes = axes.flatten()

for i, var in enumerate(valid_LV):
    ax = axes[i]
    data_v = df[var].dropna()
    ax.hist(data_v, bins=15, density=True, color='steelblue', alpha=0.6, edgecolor='white')
    # 정규분포 곡선
    x = np.linspace(data_v.min(), data_v.max(), 200)
    ax.plot(x, stats.norm.pdf(x, data_v.mean(), data_v.std()), 'r-', lw=2, label='정규분포')
    ax.set_title(f'{var}\n(M={data_v.mean():.2f}, SD={data_v.std():.2f})', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('잠재변수별 정규분포 그래프', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('정규분포_결과.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: 정규분포_결과.png')

In [ ]:
# Q-Q 플롯
fig, axes = plt.subplots(rows_n, cols_n, figsize=(5*cols_n, 4*rows_n))
axes = axes.flatten()
for i, var in enumerate(valid_LV):
    ax = axes[i]
    stats.probplot(df[var].dropna(), dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: {var}', fontsize=11)
    ax.grid(True, alpha=0.3)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Q-Q Plot (정규분포 확인)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('QQ플롯.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 5. 상관관계 분석

In [ ]:
corr = df[valid_LV].corr()

# p값 행렬
print('▶ 상관관계 분석 결과 (Pearson r)')
print('='*70)
for i, v1 in enumerate(valid_LV):
    for v2 in valid_LV[i+1:]:
        r_res = pg.corr(df[v1].dropna(), df[v2].dropna())
        r = r_res['r'].values[0]
        p = r_res['p-val'].values[0]
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
        print(f'  {v1:<15} ↔ {v2:<15}: r = {r:+.3f}, p = {p:.4f} {sig}')

print('\n  * p<.001=***, p<.01=**, p<.05=*, n.s.=유의하지않음')

In [ ]:
# 상관관계 히트맵
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r',
            mask=mask, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size':11}, linewidths=0.5)
plt.title('잠재변수 간 상관관계 히트맵', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('상관관계_히트맵.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: 상관관계_히트맵.png')

---
## STEP 6. 회귀분석

> 종속변수(Y)와 독립변수(X)를 직접 설정하세요

In [ ]:
# ✅ 여기서 독립/종속 변수 설정
X_VARS = ['내적귀인', '통제위인', '외적귀인']   # 독립변수
Y_VAR  = '행동요인_B'                           # 종속변수

reg_data = df[X_VARS + [Y_VAR]].dropna()
X = sm.add_constant(reg_data[X_VARS])
Y = reg_data[Y_VAR]
model = sm.OLS(Y, X).fit()

print(f'▶ 회귀분석: {Y_VAR} = f({" + ".join(X_VARS)})')
print('='*65)
print(f'  R²     = {model.rsquared:.3f}   (모델 설명력)')
print(f'  수정R² = {model.rsquared_adj:.3f}')
print(f'  F값    = {model.fvalue:.3f},  p = {model.f_pvalue:.4f}', '***' if model.f_pvalue<.001 else '')
print(f'  n      = {int(model.nobs)}')
print()
print(f'  {"변수":<15} {"β(비표준)":>10} {"SE":>8} {"t":>8} {"p":>10} {"유의":>6}')
print(f'  {"-"*60}')
for var in model.params.index:
    b  = model.params[var]
    se = model.bse[var]
    t  = model.tvalues[var]
    p  = model.pvalues[var]
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
    print(f'  {var:<15} {b:>10.3f} {se:>8.3f} {t:>8.3f} {p:>10.4f} {sig:>6}')

---
## STEP 7. 기울기 그래프 (회귀선 시각화)

In [ ]:
fig, axes = plt.subplots(1, len(X_VARS), figsize=(6*len(X_VARS), 5))
if len(X_VARS)==1: axes=[axes]

colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']

for i, (ax, xvar) in enumerate(zip(axes, X_VARS)):
    x = reg_data[xvar]
    y = reg_data[Y_VAR]
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)

    ax.scatter(x, y, alpha=0.4, color=colors[i%len(colors)], s=25, zorder=2)
    ax.plot(x_line, m*x_line+b, color=colors[i%len(colors)], lw=2.5,
            label=f'기울기(β) = {m:.3f}')

    # 95% 신뢰구간
    n_pts = len(x)
    se_line = y.std() * np.sqrt(1/n_pts + (x_line-x.mean())**2/((x-x.mean())**2).sum())
    ax.fill_between(x_line, m*x_line+b-1.96*se_line, m*x_line+b+1.96*se_line,
                    alpha=0.15, color=colors[i%len(colors)])

    r, p = stats.pearsonr(x, y)
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
    ax.set_xlabel(xvar, fontsize=12)
    ax.set_ylabel(Y_VAR, fontsize=12)
    ax.set_title(f'{xvar} → {Y_VAR}\nr={r:.3f}{sig}, β={m:.3f}', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'회귀분석 기울기 그래프  (종속변수: {Y_VAR})', fontsize=14)
plt.tight_layout()
plt.savefig('회귀_기울기그래프.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: 회귀_기울기그래프.png')

---
## STEP 8. 분석 결과 자동 해석 요약

In [ ]:
print('='*65)
print('  분석 결과 요약 (논문 작성용)')
print('='*65)

print(f'''
[요인분석]
  - A그룹+FCYN: 3요인 추출 (Varimax 회전)
    · 요인1(내적귀인): A1,A2,A3,A5,A6,A7,A10
    · 요인2(통제위인): FCYN1,FCYN2,FCYN3,FCYN4
    · 요인3(외적귀인): A9,A11,A12
  - B+DT+EN: 3요인 추출 (Varimax 회전)
    · 요인1(행동요인): B13,B14,B15,B16,B17
    · 요인2(동기요인): DT1,DT2,DT3,DT5,DT6
    · 요인3(환경요인): EN1~6
  - C그룹: 단일요인 구조 불명확 → 분석 제외

[신뢰도]
  - 모든 잠재변수 Cronbach α 확인 완료

[기술통계]
  - 왜도·첨도 기준(|왜도|<2, |첨도|<7) 검토
  - Shapiro-Wilk 정규성 검정 실시

[회귀분석]
  - 종속변수: {Y_VAR}
  - R² = {model.rsquared:.3f} (설명력 {model.rsquared*100:.1f}%)
  - 유의한 예측변수:''')

for var in X_VARS:
    b = model.params.get(var, float('nan'))
    p = model.pvalues.get(var, float('nan'))
    sig = '(유의)' if p < .05 else '(비유의)'
    print(f'      {var}: β={b:.3f}, p={p:.4f} {sig}')

print('''
  ⚠ 다음 단계: 분석결과 업로드 후 논거 보강 및 해석 수정''') 